# Demo: `ask()` method — Dynamic Skill Selection & Token Budget

This notebook demonstrates the `ForecastingAssistant.ask()` method with the optimizations:

1. **Dynamic skill selection** — Skills are chosen based on `task_type` + keyword matching
2. **Skill conflict resolution** — Specialized skills suppress incompatible generic ones
3. **Token budget trimming** — Skills are pruned to fit a given token budget
4. **Context-aware prompting** — Pre-computed profiles/plans are serialized into context
5. **Results mode** — Ask questions about predictions/metrics from a `forecast()` run
6. **Dynamic Ollama context sizing** — `num_ctx` is estimated from prompt size
7. **Selective reference inclusion** — `include_reference=False` saves ~7600 tokens

**Requirements:** An Ollama instance running locally (or adjust `llm=` for another provider).

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd

from skforecast_ai import ForecastingAssistant
from skforecast_ai.llm.prompts import (
    select_skills,
    estimate_prompt_tokens,
    load_skill,
    ALL_SKILLS,
    _SKILL_TOKEN_ESTIMATES,
)

warnings.filterwarnings("ignore")

## 1. Skill selection engine

Skills are selected in two phases:
- **Profile-based**: maps `task_type` → base skills
- **Keyword augmentation**: scans the question for topic patterns
- **Conflict resolution**: specialized skills suppress incompatible generic ones

In [3]:
# Single-series task with a question about prediction intervals
skills = select_skills(
    task_type="single_series",
    question="How do I get prediction intervals with conformal calibration?",
)
print("Selected skills:", skills)
print(f"Estimated tokens: {estimate_prompt_tokens(skills)}")

Selected skills: ['choosing-a-forecaster', 'forecasting-single-series', 'prediction-intervals']
Estimated tokens: 7459


In [4]:
# Multi-series + hyperparameter tuning keywords
skills = select_skills(
    task_type="multi_series",
    question="Run a bayesian search to optimize lags and n_estimators",
)
print("Selected skills:", skills)
print(f"Estimated tokens: {estimate_prompt_tokens(skills)}")

Selected skills: ['choosing-a-forecaster', 'forecasting-multiple-series', 'hyperparameter-optimization', 'autocorrelation-and-lag-selection']
Estimated tokens: 9914


In [5]:
# Foundation model question (no task_type context)
skills = select_skills(
    task_type=None,
    question="Can I use Chronos-2 for zero-shot forecasting?",
)
print("Selected skills:", skills)
print(f"Estimated tokens: {estimate_prompt_tokens(skills)}")

Selected skills: ['choosing-a-forecaster', 'foundation-forecasting']
Estimated tokens: 6221


In [6]:
# Conflict resolution: foundation keyword suppresses multi-series skill
# ForecasterFoundation handles multi-series natively (not ForecasterRecursiveMultiSeries)
skills = select_skills(
    task_type="multi_series",
    question="Can I use Chronos-2 for my multiple store series?",
)
print("Selected skills:", skills)
print("  → 'forecasting-multiple-series' is suppressed by 'foundation-forecasting'")
print(f"  Estimated tokens: {estimate_prompt_tokens(skills)}")


Selected skills: ['choosing-a-forecaster', 'foundation-forecasting']
  → 'forecasting-multiple-series' is suppressed by 'foundation-forecasting'
  Estimated tokens: 6221


In [7]:
# Without foundation keyword → multi-series skill remains
skills_no_conflict = select_skills(
    task_type="multi_series",
    question="How should I forecast multiple store series?",
)
print("No conflict:", skills_no_conflict)
print(f"  Estimated tokens: {estimate_prompt_tokens(skills_no_conflict)}")


No conflict: ['choosing-a-forecaster', 'forecasting-multiple-series']
  Estimated tokens: 4492


## 2. Token budget trimming

When a `token_budget` is specified, skills are included in order until the budget is exhausted.

In [8]:
# Show all skill sizes
print(f"{'Skill':<40} {'Tokens':>8}")
print("-" * 50)
for skill, tokens in sorted(_SKILL_TOKEN_ESTIMATES.items(), key=lambda x: x[1]):
    print(f"{skill:<40} {tokens:>8}")
print("-" * 50)
print(f"{'TOTAL':<40} {sum(_SKILL_TOKEN_ESTIMATES.values()):>8}")

Skill                                      Tokens
--------------------------------------------------
forecasting-single-series                    1224
drift-detection                              1249
feature-selection                            1407
forecasting-multiple-series                  1482
autocorrelation-and-lag-selection            2044
troubleshooting-common-errors                2108
choosing-a-forecaster                        2810
foundation-forecasting                       3211
prediction-intervals                         3225
hyperparameter-optimization                  3378
statistical-models                           3755
deep-learning-forecasting                    3883
feature-engineering                          8126
complete-api-reference                      11409
--------------------------------------------------
TOTAL                                       49311


In [9]:
# Question that matches many keywords but with a tight budget
skills_full = select_skills(
    task_type="single_series",
    question="I need ARIMA with prediction intervals and hyperparameter tuning",
    token_budget=None,  # No limit
)
print(f"No budget   -> {len(skills_full)} skills: {skills_full}")
print(f"  Estimated: {estimate_prompt_tokens(skills_full)} tokens\n")

skills_trimmed = select_skills(
    task_type="single_series",
    question="I need ARIMA with prediction intervals and hyperparameter tuning",
    token_budget=8000,  # Tight budget
)
print(f"Budget=8000 -> {len(skills_trimmed)} skills: {skills_trimmed}")
print(f"  Estimated: {estimate_prompt_tokens(skills_trimmed)} tokens")

No budget   -> 4 skills: ['choosing-a-forecaster', 'prediction-intervals', 'hyperparameter-optimization', 'statistical-models']
  Estimated: 13368 tokens

Budget=8000 -> 2 skills: ['choosing-a-forecaster', 'prediction-intervals']
  Estimated: 6235 tokens


## 3. Token estimation with/without API reference

The `include_reference` flag adds the full `llms-base.txt` (~7600 tokens). Use it only when the question needs broad API knowledge.

In [10]:
skills = ["choosing-a-forecaster", "forecasting-single-series"]

without_ref = estimate_prompt_tokens(skills, include_reference=False)
with_ref = estimate_prompt_tokens(skills, include_reference=True)

print(f"Without reference: {without_ref:,} tokens")
print(f"With reference:    {with_ref:,} tokens")
print(f"Savings:           {with_ref - without_ref:,} tokens ({100*(with_ref - without_ref)/with_ref:.0f}%)")

Without reference: 4,234 tokens
With reference:    11,834 tokens
Savings:           7,600 tokens (64%)


## 4. Full `ask()` — Q&A mode (no data)

Ask general questions about skforecast. Skills are auto-selected from keywords.

In [11]:
assistant = ForecastingAssistant(llm="ollama:qwen2.5:7b-instruct")

In [12]:
result = assistant.ask(
    prompt="What's the difference between ForecasterRecursive and ForecasterDirect?"
)
print(result.explanation)

The primary differences between `ForecasterRecursive` and `ForecasterDirect` lie in how they handle the forecast horizon, error propagation, and their suitability for different use cases.

### Recursive Forecaster

- **Models Trained**: 1 model is trained (one estimator).
- **Error Propagation**: Predictions from one step feed into the next step. This means errors can accumulate over time.
- **Forecast Horizon**: Flexible; you can specify any number of steps at prediction time using `steps` parameter.
- **Best For**: 
  - Most cases, especially when dealing with medium to short forecast horizons where error propagation is manageable.
  - When the predictive relationships are somewhat consistent across the entire horizon.

### Direct Forecaster

- **Models Trained**: One model per step of the forecast (N models in total for a multi-step forecast).
- **Error Propagation**: No error propagation. Each model's predictions do not depend on previous steps' predictions.
- **Forecast Horizon**:

In [13]:
# Keyword triggers foundation-forecasting skill
result = assistant.ask(
    prompt="How do I use Chronos-2 for multi-series zero-shot forecasting?"
)
print(result.explanation)

To use Chronos-2 for multi-series zero-shot forecasting in a zero-shot manner (i.e., without retraining), you can follow these steps:

### Step-by-Step Guide

1. **Install the Required Libraries**:
   Ensure you have installed the necessary libraries and the specific model variant for Chronos-2.

   ```bash
   pip install chronos-forecasting -qqq  # Install Chronos-forecasting library
   ```

2. **Define Your Multi-Series Data**:
   Prepare your data in a way that each series is a column or can be indexed appropriately.

3. **Initialize the `FoundationModel` with ChronosAdapter**:
   Use the `AutogluonChronosPipeline` to initialize the model.

4. **Fit (Store Context) Without Training**:
   Call `fit` on your multi-series data; this will store the context but not train the model.

5. **Predict Using Different Windows if Needed**:
   You can override the stored context using the `context` parameter in the `predict` method.

6. **Backtesting (Optional)**:
   Use backtesting functions to 

## 5. Full `ask()` — Explain mode (with data)

When data is provided, the assistant runs the deterministic profiling + plan pipeline first, then explains the result via the LLM.

In [14]:
# Create sample data
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)

df = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": rng.normal(50, 10, n),
})
df.head()

,date,sales,promo
0,2022-01-01,100.304717,64.814555
1,2022-01-02,103.173890,42.564118
2,2022-01-03,104.889824,41.777500
3,2022-01-04,103.125168,52.023062
4,2022-01-05,96.835295,58.443852


In [15]:
result = assistant.ask(
    prompt="Explain the forecasting strategy for this dataset",
    data=df,
    target="sales",
    date_column="date",
    steps=14,
)

print("=" * 60)
print("PROFILE")
print("=" * 60)
print(f"Task type: {result.forecaster_profile.task_type}")
print(f"Forecaster: {result.forecaster_profile.forecaster}")
print(f"Estimator: {result.forecaster_profile.estimator}")

print("\n" + "=" * 60)
print("PLAN")
print("=" * 60)
print(f"Explanation: {result.plan.explanation}")

print("\n" + "=" * 60)
print("LLM EXPLANATION")
print("=" * 60)
print(result.explanation)

PROFILE
Task type: single_series
Forecaster: ForecasterRecursive
Estimator: LGBMRegressor

PLAN
Explanation: Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 3, 7, 8, 10, 12, 118]. Window features: ['mean', 'mean']. NaN rows kept (NaN-tolerant estimator). Exogenous variables included.

LLM EXPLANATION
### Forecasting Strategy for the Dataset

Given the profile, we will use a single-series ML forecaster to predict future sales values in our time series data. Here's the detailed strategy:

#### 1. **Forecaster Type**
We select `ForecasterRecursive` as the primary forecaster because it handles multi-step ahead forecasting by using its own predictions from previous steps as input features for subsequent forecasts. This is particularly useful for datasets with complex, dynamic patterns.

Alternative forecasters like `ForecasterDirect`, which creates separate models for each step ahead, and `ForecasterFoundation` (which uses a pre-trained model), may also be considered depending on the s

## 6. Explicit skill override

Override auto-selection to force specific skills (e.g., troubleshooting).

In [16]:
result = assistant.ask(
    prompt="I'm getting a ValueError when calling predict(). What should I check?",
    skills=["troubleshooting-common-errors", "complete-api-reference"],
)
print(result.explanation)

Certainly! When you encounter a `ValueError` while using the `predict()` method, there are several potential issues that could be causing the error. Here are some steps to troubleshoot and resolve common issues:

1. **Check Input Parameters:**
   - Ensure that the `n_periods` parameter is correctly set. This should represent the number of future periods you want to predict.
   - Verify that any required exogenous variables (`exog`) passed in match the expected dimensions.

2. **Parameter Types and Compatibility:**
   - Confirm that all input parameters are of the correct type (e.g., `n_periods` is an integer).
   - Check for compatibility issues between different forecasters and their specific requirements (e.g., multivariate forecaster vs. univariate).

3. **Model Fitting:**
   - Make sure that your forecaster has been properly fitted on training data using the `fit()` method before calling `predict()`.
   - If you are working with a recursive model, ensure that initial training data 

## 7. Include full API reference

When the question needs broad API knowledge, set `include_reference=True` (adds ~7600 tokens).

In [17]:
result = assistant.ask(
    prompt="List all available datasets in skforecast and their frequencies",
    skills=["complete-api-reference"],
    include_reference=True,
)
print(result.explanation)

Certainly! To list all the available datasets along with their frequencies in `skforecast`, you can use the `show_datasets_info()` function. Below is a step-by-step guide on how to do this:

1. **Import necessary functions**: You'll need to import `fetch_dataset` and `show_datasets_info` from `skforecast.datasets`.

2. **Use `show_datasets_info()`**: This function will print out information about all available datasets, including their names and frequencies.

Here is the code you can run in your Python environment:

```python
# Import the required functions
from skforecast.datasets import fetch_dataset, show_datasets_info

# Display information about all available datasets
show_datasets_info()
```

When you run this code, it will output a detailed list of all datasets along with their frequencies. For example, some common datasets and their frequencies might look like:

```plaintext
+------------------------------------------------+-----------+
|            Dataset Name                

## 8. Multi-series explain mode

The assistant detects multi-series data and loads relevant skills automatically.

In [18]:
# Multi-series long-format data
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "store": ["A"] * n_multi + ["B"] * n_multi + ["C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
})

result = assistant.ask(
    prompt="What approach will you use for these multiple store series?",
    data=df_multi,
    target="revenue",
    date_column="date",
    series_id_column="store",
    steps=7,
)

print(f"Task type: {result.forecaster_profile.task_type}")
print(f"Forecaster: {result.forecaster_profile.forecaster}")
print(f"\n{result.explanation}")

Task type: multi_series
Forecaster: ForecasterRecursiveMultiSeries

Given the profile of your dataset, which has three revenue time series and 200 observations with a daily frequency, we recommend using `ForecasterRecursiveMultiSeries` to model these series simultaneously. This approach is suitable when the series are related or share common patterns that can be learned by a single global model.

Here’s why:

1. **Multiple Time Series**: There are three revenue time series, and each needs forecasting.
2. **Daily Frequency Data**: A daily frequency means we have enough temporal information to capture short-term patterns effectively.

### Plan

We will use `ForecasterRecursiveMultiSeries` with the following configuration:

- **Estimator**: `LGBMRegressor`
- **Steps**: 7 (multi-step forecasting is needed)
- **lags**: `[1, 7, 10, 14, 36, 48, 55]` to capture different temporal patterns.
- **Window Features**: `['mean', 'mean']` to incorporate rolling mean values which help in capturing rece

## 9. Results mode — Ask about forecast predictions

After running `forecast()`, pass the `RunResult` to `ask()` so the LLM can explain actual predictions, metrics, and intervals.

In [19]:
# Run a forecast first
forecast_result = assistant.forecast(
    data=df,
    target="sales",
    date_column="date",
    steps=14,
)

print(f"Predictions shape: {forecast_result.predictions.shape}")
print(f"Metrics:\n{forecast_result.metrics.to_string(index=False)}")
print(f"\nPredictions:\n{forecast_result.predictions}")


Predictions shape: (14, 1)
Metrics:
series      MAE      MSE     MASE
 sales 0.859057 1.118199 0.300112

Predictions:
                 pred
2022-10-20  85.738240
2022-10-21  85.611453
2022-10-22  88.351413
2022-10-23  93.152032
2022-10-24  94.301703
2022-10-25  91.164140
2022-10-26  86.611982
2022-10-27  84.629980
2022-10-28  83.968107
2022-10-29  87.719652
2022-10-30  93.398525
2022-10-31  94.460990
2022-11-01  92.090153
2022-11-02  86.346873


In [20]:
# Ask the LLM about the forecast results
answer = assistant.ask(
    prompt="Explain the predictions. What trend do you see and what values are expected?",
    run_result=forecast_result,
)
print(answer.explanation)


### Explanation of Predictions

Based on the provided forecast results, we can analyze the trends and values anticipated for the next 14 days starting from October 20, 2022.

#### Overall Trend:
- **Trend Analysis**: The predicted sales values show a general upward trend over the first few days (October 20 to October 27), with higher sales on October 23 and October 25. However, there are fluctuations between these high points.
- **Decline in Trends**: After October 27, the forecasted sales start to decrease slightly but continue rising again from October 30 onwards.

#### Specific Values:

1. **October 20 – 27**:
   - The sales values increase progressively during this period. The highest value is observed on October 25 at approximately 94.3 units.

2. **October 28 – November 2**:
   - There is a slight decrease in the forecasted sales from October 28 to November 1, after which there's another gradual increase.
   - On November 1, the predicted sales value comes down slightly but then 

In [21]:
# The result also carries the deterministic code from the forecast
print("Code available:", answer.code is not None)
print(f"Code length: {len(answer.code)} chars")
print(f"\nFirst 5 lines:\n{chr(10).join(answer.code.splitlines()[:5])}")


Code available: True
Code length: 2029 chars

First 5 lines:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from skforecast.metrics import mean_absolute_scaled_error
from skforecast.preprocessing import RollingFeatures
